In [168]:
import time

timings = {}

In [169]:
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
if (cwd / "src").exists():
    project_root = cwd
elif (cwd / "integration.py").exists():
    project_root = cwd.parent
else:
    project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
print("Project root:", project_root)

Project root: c:\Users\Acer\Documents\RAG-from-scratch\RAG-from-scratch


In [170]:
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage


from langchain_ollama import ChatOllama, OllamaEmbeddings


from data_source.wikipedia import WikipediaSource
from data_source.web_search import WebSearchSource


from src.pipeline.indexing_rag import build_vectorstore
from src.pipeline.retrieval_rag import create_retriever


from src.query_translation.multi_query import (
    create_multi_query_retrieval_chain,
)

from src.Reranking.reranking import CrossEncoderReranker


from src.advanced_indexing.raptor import RaptorIndexer


from src.advanced_RAG.Self_RAG.self_rag import SelfRAG
from src.advanced_RAG.Long_Context.long_context import LongContext


from src.memory.memory import ConversationMemory


from src.evaluation.rag_evaluation import RAGEvaluator

In [171]:
# Load environment variables

load_dotenv()


# Local LLM

llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
    base_url="http://127.0.0.1:11434",
)


# Local embedding model

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://127.0.0.1:11434",
)

In [172]:
# Final generation prompt

generation_prompt = ChatPromptTemplate.from_template("""
You are a careful RAG assistant.

Answer the question using ONLY the provided evidence.

Evidence:
{context}

Question:
{question}

Rules:
- Do not invent facts.
- Use only the provided evidence.
- For current or time-sensitive questions, prefer the newest
  evidence with an explicit date and time.
- If sources conflict, prefer the most recent dated evidence.
- Do not use older information when newer evidence is available.
- If the evidence is insufficient, say that the information
  is not available in the retrieved evidence.
- Answer clearly and concisely.

Answer:
""")

generation_chain = generation_prompt | llm | StrOutputParser()

In [173]:
# Data sources

wikipedia = WikipediaSource(
    top_k=5,
)

web_search = WebSearchSource(
    top_k=5,
)


# Reranker

reranker = CrossEncoderReranker(
    top_k=5,
)


# Conversation memory

conversation_memory = ConversationMemory()


# Evaluator

evaluator = RAGEvaluator(
    model="llama3:latest",
    temperature=0,
)

In [174]:
# User query
start = time.perf_counter()

query = input("Ask a question: ")

# Detect current / time-sensitive queries


def is_current_query(query: str) -> bool:
    keywords = [
        "today",
        "current",
        "latest",
        "now",
        "live",
        "recent",
        "yesterday",
        "tomorrow",
    ]

    query_lower = query.lower()

    return any(keyword in query_lower for keyword in keywords)


current_query = is_current_query(query)

print("Query:", query)
print("Current query:", current_query)


# Determine retrieval path

if current_query:
    decision = "NOT_ANSWERABLE"
    print("→ Current query: External RAG required.")

else:
    decision = "ANSWERABLE"
    print("→ Stable query: Answer directly with Ollama.")


timings["query processing"] = time.perf_counter() - start

print(f"Query Processing: {timings['query processing']:.2f}s")

Query: what is my name ?
Current query: False
→ Stable query: Answer directly with Ollama.
Query Processing: 8.82s


In [175]:
# External retrieval
start = time.perf_counter()
documents = []

if decision == "NOT_ANSWERABLE":

    if current_query:

        # Current / time-sensitive information
        documents = web_search.retrieve(query)

        print("Source: Tavily Web Search")

        print(
            "Documents:",
            len(documents),
        )

    else:

        # Stable external information
        wikipedia_documents = wikipedia.retrieve(query)
        web_documents = web_search.retrieve(query)

        documents = wikipedia_documents + web_documents

        print(
            "Wikipedia:",
            len(wikipedia_documents),
        )

        print(
            "Web:",
            len(web_documents),
        )

        print(
            "Total:",
            len(documents),
        )

        timings["external retrieval"] = time.perf_counter() - start
        print(f"External Retrieval: {timings['external retrieval']:.2f}s")

In [176]:
# Deduplicate external documents
start = time.perf_counter()


if decision == "NOT_ANSWERABLE":

    unique_documents = {}

    for document in documents:

        url = document.metadata.get("url")

        if url:
            key = url
        else:
            key = document.metadata.get("title", "") + document.page_content[:200]

        if key not in unique_documents:
            unique_documents[key] = document

    documents = list(unique_documents.values())

    print(
        "Unique documents:",
        len(documents),
    )
    timings["deduplication"] = time.perf_counter() - start
    print(f"Deduplication: {timings['deduplication']:.2f}s")

In [177]:
for i, document in enumerate(documents, 1):
    print(
        i,
        document.metadata.get("title"),
        document.metadata.get("url"),
    )

In [178]:
# Reranking
if decision == "NOT_ANSWERABLE":
    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=documents,
    )

In [179]:
from datetime import datetime


def extract_document_datetime(document):
    """Extract document datetime from metadata."""
    metadata = getattr(document, "metadata", {}) or {}

    # Try common metadata keys
    for key in ["date", "datetime", "created_at", "published_at", "timestamp"]:
        value = metadata.get(key)

        if value is not None:
            return value

    return None

In [180]:
# Inspect ranking

if decision == "NOT_ANSWERABLE":

    for i, document in enumerate(
        reranked_documents,
        1,
    ):
        print(f"\n--- Rank {i} ---")

        print(
            "Title:",
            document.metadata.get("title"),
        )

        print(
            "URL:",
            document.metadata.get("url"),
        )

        print(
            "Date:",
            extract_document_datetime(document),
        )

        print(document.page_content[:700])

In [181]:
# Build vector store

vectorstore = None
retriever = None
multi_query_retriever = None

if decision == "NOT_ANSWERABLE":

    vectorstore = build_vectorstore(
        documents=documents,
        embedding_model=embeddings,
        batch_size=32,
    )

    print("Vector store created.")

In [182]:
# Create retriever

if decision == "NOT_ANSWERABLE":

    retriever = create_retriever(
        vectorstore=vectorstore,
        k=5,
    )

In [183]:
# Multi-Query retrieval
start = time.perf_counter()


if decision == "NOT_ANSWERABLE":

    multi_query_retriever = create_multi_query_retrieval_chain(
        retriever=retriever,
        llm=llm,
    )

    retrieved_documents = multi_query_retriever.invoke(query)

    print(
        "Retrieved documents:",
        len(retrieved_documents),
    )
    timings["multi-query retrieval"] = time.perf_counter() - start
    print(f"Multi-Query Retrieval: {timings['multi-query retrieval']:.2f}s")

In [184]:
# Rerank stable RAG results

if decision == "NOT_ANSWERABLE":

    reranked_documents = reranker.rerank_documents(
        query=query,
        documents=retrieved_documents,
    )

    print(
        "Stable RAG reranked documents:",
        len(reranked_documents),
    )

In [185]:
# RAPTOR
start = time.perf_counter()

raptor_leaf = []
raptor_clusters = []

if decision == "NOT_ANSWERABLE":

    raptor = RaptorIndexer(
        llm=llm,
        embeddings=embeddings,
        n_clusters=3,
    )

    raptor.build_tree(
        documents=documents,
    )

    raptor_results = raptor.retrieve(
        query=query,
        k=3,
    )

    raptor_leaf = raptor_results.get(
        "leaf",
        [],
    )

    raptor_clusters = raptor_results.get(
        "clusters",
        [],
    )

    print(
        "RAPTOR leaf results:",
        len(raptor_leaf),
    )

    print(
        "RAPTOR cluster results:",
        len(raptor_clusters),
    )
timings["raptor"] = time.perf_counter() - start
print(f"RAPTOR: {timings['raptor']:.2f}s")

RAPTOR: 0.00s


In [186]:
# Self-RAG
start = time.perf_counter()

self_rag_answer = ""

if decision == "NOT_ANSWERABLE":

    self_rag = SelfRAG(
        llm=llm,
        retriever=retriever,
    )

    self_rag_answer = self_rag.invoke(
        query,
        max_retries=2,
    )

    print("Self-RAG completed.")
    timings["self_rag"] = time.perf_counter() - start
    print(f"Self-RAG: {timings['self_rag']:.2f}s")

In [187]:
# Long-context processing
start = time.perf_counter()
long_context_answer = ""

if decision == "NOT_ANSWERABLE":

    long_context = LongContext(
        model="llama3:latest",
    )

    long_context_result = long_context.run(
        documents=reranked_documents,
        query=query,
        compress=True,
    )

    if isinstance(
        long_context_result,
        dict,
    ):
        long_context_answer = long_context_result.get(
            "answer",
            "",
        )
    else:
        long_context_answer = str(long_context_result)

timings["long_context"] = time.perf_counter() - start
print(f"Long-Context Processing: {timings['long_context']:.2f}s")

Long-Context Processing: 0.00s


In [188]:
# Build final context
start = time.perf_counter()
context = ""

if decision == "NOT_ANSWERABLE":

    reranked_context = "\n\n".join(
        document.page_content for document in reranked_documents
    )

    if current_query:

        # Current/live queries:

        context = reranked_context

    else:

        # Stable external RAG:

        context_parts = [
            reranked_context,
        ]

        if raptor_leaf:
            context_parts.append(
                "RAPTOR Leaf Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_leaf)
            )

        if raptor_clusters:
            context_parts.append(
                "RAPTOR Cluster Evidence:\n"
                + "\n\n".join(str(item) for item in raptor_clusters)
            )

        if self_rag_answer:
            context_parts.append("Self-RAG Evidence:\n" + self_rag_answer)

        if long_context_answer:
            context_parts.append("Long-Context Evidence:\n" + long_context_answer)

        context = "\n\n".join(context_parts)

    print(
        "Final context length:",
        len(context),
    )
    timings["final_context"] = time.perf_counter() - start
    print(f"Final Context Building: {timings['final_context']:.2f}s")

In [189]:
# Final answer generation
start = time.perf_counter()
if decision == "ANSWERABLE":

    response = llm.invoke(query)
    answer = response.content

else:

    answer = generation_chain.invoke(
        {
            "context": context,
            "question": query,
        }
    )
    timings["final_answer"] = time.perf_counter() - start
    print(f"Final Answer Generation: {timings['final_answer']:.2f}s")

In [190]:
# Store conversation memory

conversation_memory.add_message(HumanMessage(content=query))

conversation_memory.add_message(AIMessage(content=answer))

In [191]:
# Evaluation
start = time.perf_counter()
if decision == "ANSWERABLE":

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "answer_relevance": answer_relevance,
    }

else:

    context_relevance = evaluator.evaluate_context_relevance(
        question=query,
        context=context,
    )

    faithfulness = evaluator.evaluate_faithfulness(
        context=context,
        answer=answer,
    )

    answer_relevance = evaluator.evaluate_answer_relevance(
        question=query,
        answer=answer,
    )

    evaluation_results = {
        "context_relevance": context_relevance,
        "faithfulness": faithfulness,
        "answer_relevance": answer_relevance,
    }

print("Evaluation:")
for metric, value in evaluation_results.items():
    print(f"{metric}: {value}")

timings["evaluation"] = time.perf_counter() - start
print(f"Evaluation: {timings['evaluation']:.2f}s")

Evaluation:
answer_relevance: RELEVANT
Evaluation: 1.04s


In [192]:
print("\n" + "=" * 60)
print("PIPELINE LATENCY")
print("=" * 60)

for stage, duration in timings.items():
    print(f"{stage:<25} {duration:.2f}s")

print("=" * 60)

total_time = sum(timings.values())

print(f"Total Pipeline Time: {total_time:.2f}s")


PIPELINE LATENCY
query processing          8.82s
raptor                    0.00s
long_context              0.00s
evaluation                1.04s
Total Pipeline Time: 9.86s


In [193]:
# Final output

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)

print(answer)

print("=" * 60)

print(
    "\nDecision:",
    decision,
)

print(
    "Mode:",
    (
        "Direct Ollama"
        if decision == "ANSWERABLE"
        else ("Current Web RAG" if current_query else "Advanced RAG")
    ),
)


FINAL ANSWER
I apologize, but I don't have any information about your name. I'm a large language model, I don't have personal knowledge or access to personal information about individuals. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to learn it and address you by name in our conversation!

Decision: ANSWERABLE
Mode: Direct Ollama


In [194]:
# Prepare evaluation scores for visualization

evaluation_scores = {
    metric: float(value)
    for metric, value in evaluation_results.items()
    if isinstance(value, (int, float))
}

print("Evaluation Scores:")

for metric, score in evaluation_scores.items():
    print(f"{metric}: {score}")

Evaluation Scores:


In [195]:
print(evaluation_results)

{'answer_relevance': 'RELEVANT'}


In [196]:
import matplotlib.pyplot as plt

# Convert evaluator outputs into binary scores
evaluation_scores = {
    "Context Relevance": (
        1 if evaluation_results["context_relevance"].startswith("RELEVANT") else 0
    ),
    "Faithfulness": (
        1 if evaluation_results["faithfulness"].startswith("FAITHFUL") else 0
    ),
    "Answer Relevance": (
        1 if evaluation_results["answer_relevance"].startswith("RELEVANT") else 0
    ),
}

metrics = list(evaluation_scores.keys())
scores = list(evaluation_scores.values())

plt.figure(figsize=(8, 5))

plt.bar(metrics, [score * 100 for score in scores])

plt.xlabel("Evaluation Metric")
plt.ylabel("Score (%)")
plt.title("RAG Pipeline Evaluation")

plt.ylim(0, 100)

plt.show()

# Overall score
overall_score = (sum(scores) / len(scores)) * 100

print(f"Overall Evaluation Score: {overall_score:.1f}%")

KeyError: 'context_relevance'